# Hyperparameter Tuning via METIS

## Hyper Band

### Loading Libraries

In [ ]:
# Numerical Computing
import math
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# StatsModel
from scipy.stats import randint,truncnorm,uniform

# Scikit-Learn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, make_scorer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score

# OS
import time
import random
import operator

# DEAP
from deap import base
from deap import tools
from deap import creator

#
import multiprocessing

In [ ]:
sns.set()

### Loading Data

In [ ]:
df = pd.read_csv("/Users/joaquinromero/Desktop/HPTP/data/train.csv", sep=";")

In [ ]:
df['y'] = df['y'].map({'yes':1,'no':0})

### Train/Test Split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.1, random_state=0) 

#### Placing Numerical Features

In [ ]:
numerical_feats = list(df_train.drop(columns='y').select_dtypes(include=np.number).columns)

#### Placing Categorical Features

In [ ]:
categorical_feats = list(df_train.drop(columns='y').select_dtypes(exclude=np.number).columns)

### Preprocessor

In [ ]:
# Normalization Pre-processing for Numerical Features
numeric_preprocessor = StandardScaler()

# One-Hot-Encoding Pre-processing for Categorical Features
categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numerical_feats),
        ("cat", categorical_preprocessor, categorical_feats),
    ]
)

### Pipeline

In [ ]:
pipe = Pipeline(
    steps=[("preprocessor", preprocessor), 
           ("model", RandomForestClassifier(random_state=0))]
)

#### Placing All Features for Training Set

In [ ]:
X_train_full = df_train.drop(columns=['y'])
y_train = df_train['y']

X_train_full.info()

#### Placing All Features for Test Set

In [ ]:
X_test_full = df_test.drop(columns=['y'])
y_test = df_test['y']

X_test_full.info()

#### Calculate F1-Score on Test Data without Hyperparameter Tuning

In [ ]:
# Fit the pipeline on train data 
pipe.fit(X_train_full,y_train)

# Evaluate on the test data 
y_pred = pipe.predict(X_test_full)
print(f1_score(y_test, y_pred))

### Hyperparameter Space: METIS Hyper Band

In [ ]:
experiment = Experiment('local')

experiment.config.experiment_name = 'nni_sklearn_hyper_band'
experiment.config.advisor.name = 'Hyperband'
experiment.config.advisor.class_args['optimize_mode'] = 'maximize'
experiment.config.advisor.class_args['R'] = 200
experiment.config.advisor.class_args['eta'] = 3
experiment.config.advisor.class_args['exec_mode'] = 'parallelism'

# Boilerplate code
experiment.config.trial_command = "python '/Users/joaquinromero/Desktop/HPTP/nni/model_advisor.py'"  #please change this to your own local path
experiment.config.trial_code_directory = '.'
experiment.config.search_space = hyperparameter_space_advisor
experiment.config.max_trial_number = 100
experiment.config.trial_concurrency = 10
experiment.config.max_experiment_duration = '1h'

In [ ]:
experiment.run(8080, wait_completion = True, debug = False)

In [ ]:
best_trial = sorted(experiment.export_data(),key = lambda x: x.value, reverse = True)[0]

best_trial.parameter['model__n_estimators'] = best_trial.parameter['TRIAL_BUDGET'] * 50
del best_trial.parameter['TRIAL_BUDGET']

print(best_trial.value, best_trial.parameter)

In [ ]:
tuned_pipe = clone(pipe).set_params(**best_trial.parameter)

# Fit the pipeline on train data 
tuned_pipe.fit(X_train_full,y_train)

# Evaluate on the test data 
y_pred = tuned_pipe.predict(X_test_full)
print(f1_score(y_test, y_pred))

In [ ]:
experiment.stop()